# Appendix A1: MCMC diagnostics

Companion notebook. The figures are produced by `generate.py`; this notebook walks the diagnostics one at a time so you can re-run any single check without rebuilding all three figures.

Reference: [chapter.md](chapter.md).

In [ ]:
import arviz as az
import numpy as np
import pymc as pm
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
az.style.use('arviz-darkgrid')

## Loop A: a healthy sampler

Refit the Chapter 6 coin model and read the four diagnostics arviz computes by default.

In [ ]:
rng = np.random.default_rng(7)
seq = rng.binomial(1, 0.55, size=200)
with pm.Model():
    p = pm.Beta('p', alpha=1, beta=1)
    pm.Binomial('obs', n=len(seq), p=p, observed=int(seq.sum()))
    idata = pm.sample(2000, tune=1000, chains=4, random_seed=606, progressbar=False)
az.summary(idata, var_names=['p'])

In [ ]:
az.plot_trace(idata, var_names=['p']);

## Loop B: the funnel

The same model in two parameterizations: centered (hostile geometry) and non-centered (well-conditioned). With only 8 latent groups and no data this is a demonstration of how the parameterization alone determines whether the sampler can mix.

In [ ]:
with pm.Model() as centered:
    mu = pm.Normal('mu', mu=0, sigma=1)
    tau = pm.HalfNormal('tau', sigma=1)
    pm.Normal('theta', mu=mu, sigma=tau, shape=8)
    idata_c = pm.sample(2000, tune=1500, chains=4, random_seed=11, progressbar=False, target_accept=0.9)
with pm.Model() as noncentered:
    mu = pm.Normal('mu', mu=0, sigma=1)
    tau = pm.HalfNormal('tau', sigma=1)
    z = pm.Normal('z', mu=0, sigma=1, shape=8)
    pm.Deterministic('theta', mu + tau * z)
    idata_nc = pm.sample(2000, tune=1500, chains=4, random_seed=11, progressbar=False, target_accept=0.9)
for name, ida in [('centered', idata_c), ('non-centered', idata_nc)]:
    div = int(ida.sample_stats['diverging'].sum().item())
    rhat_tau = float(az.rhat(ida, var_names=['tau']).tau.values)
    ess_tau = float(az.ess(ida, var_names=['tau']).tau.values)
    print(f'{name:14s}  divergences={div:4d}  R-hat(tau)={rhat_tau:.3f}  ESS(tau)={ess_tau:.0f}')

## Loop C: how the diagnostics move with run length

Sample the healthy coin model at increasing draw counts and watch R-hat shrink toward 1 and ESS grow roughly linearly.

In [ ]:
draws_grid = [200, 400, 800, 1500, 3000]
for d in draws_grid:
    with pm.Model():
        p = pm.Beta('p', alpha=1, beta=1)
        pm.Binomial('obs', n=len(seq), p=p, observed=int(seq.sum()))
        ida = pm.sample(d, tune=500, chains=4, random_seed=606 + d, progressbar=False)
    rhat = float(az.rhat(ida).p.values)
    ess = float(az.ess(ida).p.values)
    print(f'draws={d:5d}  R-hat={rhat:.4f}  ESS={ess:.0f}')